# Mashtots: классификация армянских рукописных букв (78 классов)

Соревнование: [Mashtots Dataset](https://www.kaggle.com/c/mashtots-dataset).
Train — 70 060 изображений **64×64, grayscale**, разложенных по 78 папкам
(39 букв армянского алфавита × заглавная/строчная).

## Диагноз исходной версии: модель не обучалась вообще

Лог обучения исходного ноутбука:

```
Epoch 1/5  accuracy: 0.0125 - loss: 4.3572 - val_accuracy: 0.0119 - val_loss: 4.3566
Epoch 2/5  accuracy: 0.0126 - loss: 4.3565 - val_accuracy: 0.0119 - val_loss: 4.3568
```

Это не «плохо обучилось», это ровно константный выход:

* `ln(78) = 4.35671` — кросс-энтропия равномерного распределения по 78 классам;
* `1/78 = 0.01282` — accuracy случайного угадывания.

Loss стоит на `4.3565` с первой эпохи и не двигается → сеть выдаёт почти
равномерный softmax независимо от картинки. Ниже (раздел «Диагностика»)
это воспроизведено численно.

## Что исправлено

| Проблема в исходнике | Последствие | Исправление |
|---|---|---|
| Голова `Dense(32) → Dense(10, relu) → Dense(2, relu) → Dense(78, softmax)` | 78 классов протискиваются через **2 ReLU-нейрона**; градиент через два подряд узких ReLU вырождается, мёртвая единица из двух убивает половину представления → равномерный softmax | бутылочное горло убрано: `Flatten → Dense(256) → BN → Dropout → Dense(78)` |
| Пиксели не нормализованы (`cv2.imread` → uint8 `0..255`) | при `Flatten` 200×200 предактивации порядка `1e4`, первый шаг Adam с `lr=1e-3` «убивает» ReLU | слой `Rescaling(1/255)` **внутри** модели |
| `cv2.resize(image, (200, 200))` при родных 64×64 | ×9.8 пикселей и вычислений, нуль новой информации; `X` = 2.6 ГиБ uint8 (10.4 ГиБ во float32); `Flatten` → 153 664 признака → `Dense(128)` = 19.7 М параметров в одном слое (97 % модели) | работаем в родных 64×64 |
| `train_test_split` без `stratify` | перекос классов между train/test | `stratify=y` на обоих разбиениях |
| Тест использовался как `validation_data` | подбор модели по тесту — утечка, честной оценки нет | три части: train / val / test |
| `to_categorical(..., num_classes=78)` | хардкод числа классов + лишняя матрица `N×78` | `sparse_categorical_crossentropy` по целым меткам |
| `input_shape=` в `Conv2D` | `UserWarning` в Keras 3 | `keras.Input(shape=...)` |
| Нет BatchNorm, аугментации, колбэков; ровно 5 эпох | недообучение и никакой защиты от переобучения | BN, `Random*`-аугментация, `EarlyStopping` + `ReduceLROnPlateau` + `ModelCheckpoint` |
| `os.listdir` без сортировки, без фильтра расширений, без проверки `imread() is None` | недетерминированный порядок, падение на `.ipynb_checkpoints` и битых файлах | сортировка, белый список расширений, пропуск нечитаемых |
| `mpimg.imread('')` | пустой путь → исключение | параметр пути + понятная ошибка |
| `cv2.cvtColor(img, cv2.IMREAD_GRAYSCALE)` | `IMREAD_GRAYSCALE == 0` — это **не** код цветового преобразования (0 = `COLOR_BGR2BGRA`); к тому же вызвано уже после `resize` | чтение сразу через `cv2.imread(..., IMREAD_GRAYSCALE)`, `cvtColor` не нужен |
| `model.predict(...)` без `argmax`, без нормализации | на выходе 78 вероятностей вместо класса + train/serve skew | `predict_letter()` — один общий путь предобработки с обучением |
| Ячейки `X`, `X_train`, `X_test` печатали массивы целиком | мегабайты вывода в `.ipynb` | печатаем `shape` / `dtype` / объём в памяти |
| Нет метрик и submission | непонятно, что получилось, и нечего отправить | `classification_report`, confusion matrix, разбор ошибок, `submission.csv` |

В разделе 6 разобрана ещё одна ловушка того же семейства: `BatchNormalization`
с дефолтным `momentum=0.99` даёт внешне такую же картину «accuracy = 1/78», но
только на валидации, и лечится совсем иначе.

## 1. Импорты и конфигурация

Все параметры собраны в одном месте и переопределяются переменными окружения —
удобно для быстрого прогона (`MASHTOTS_EPOCHS=2 MASHTOTS_MAX_PER_CLASS=20`).

In [ ]:
import os
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

SEED = 42
IMG_SIZE = 64          # родное разрешение датасета: апскейлить нечего
NUM_CLASSES = 78
BATCH_SIZE = 128
EPOCHS = int(os.environ.get("MASHTOTS_EPOCHS", 30))
DATA_DIR = Path(os.environ.get("MASHTOTS_DATA", "data/mashtots"))
MAX_PER_CLASS = int(os.environ.get("MASHTOTS_MAX_PER_CLASS", 0)) or None
RUN_DIAGNOSTIC = os.environ.get("MASHTOTS_DIAGNOSTIC", "1") == "1"
MODEL_PATH = Path("mashtots_cnn.keras")

keras.utils.set_random_seed(SEED)
rng = np.random.default_rng(SEED)
sns.set_theme(style="whitegrid")

print("tensorflow", tf.__version__, "| keras", keras.__version__)
print("устройства:", [d.device_type for d in tf.config.list_physical_devices()])

## 2. Загрузка данных

Отличия от исходной версии:

* каталог с классами ищется автоматически (`Train/Train`, `Train`, сам `DATA_DIR`);
* классы и файлы перебираются **отсортированными** → воспроизводимый порядок;
* массив выделяется заранее (`np.empty`) вместо `list` + `np.array` — нет удвоения памяти на копии;
* храним `uint8`, а не `float32`: 274 МиБ против 1.07 ГиБ. Keras сам приведёт батч к float,
  а нормализацию делает слой `Rescaling` внутри модели;
* нечитаемые файлы и посторонние расширения пропускаются, а не роняют цикл.

In [ ]:
IMAGE_EXT = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff", ".pgm"}


def list_class_dirs(root: Path) -> list[Path]:
    return sorted(
        (p for p in root.iterdir() if p.is_dir() and p.name.isdigit()),
        key=lambda p: int(p.name),
    )


def list_images(directory: Path) -> list[Path]:
    return sorted(p for p in directory.iterdir() if p.is_file() and p.suffix.lower() in IMAGE_EXT)


def find_class_root(base: Path) -> Path:
    """Каталог, внутри которого лежат папки-классы с числовыми именами."""
    seen = set()
    for cand in (base, base / "Train", base / "Train" / "Train", Path("Train"), Path("Train") / "Train"):
        if cand in seen:
            continue
        seen.add(cand)
        if cand.is_dir() and len(list_class_dirs(cand)) > 1:
            return cand
    raise FileNotFoundError(
        f"не найдены папки-классы. Скачайте датасет соревнования и распакуйте так, "
        f"чтобы существовал путь {base}/Train/<номер класса>/*.png"
    )


def load_dataset(root: Path, img_size: int = IMG_SIZE, max_per_class: int | None = None):
    class_dirs = list_class_dirs(root)
    samples = []
    for cdir in class_dirs:
        samples += [(f, int(cdir.name)) for f in list_images(cdir)[:max_per_class]]

    X = np.empty((len(samples), img_size, img_size, 1), dtype=np.uint8)
    y = np.empty(len(samples), dtype=np.int16)

    n, resized, skipped = 0, 0, 0
    for path, label in samples:
        img = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
        if img is None:
            skipped += 1
            continue
        if img.shape != (img_size, img_size):
            img = cv2.resize(img, (img_size, img_size), interpolation=cv2.INTER_AREA)
            resized += 1
        X[n, :, :, 0] = img
        y[n] = label
        n += 1

    print(f"классов: {len(class_dirs)} | загружено: {n} | ресайз: {resized} | пропущено: {skipped}")
    return X[:n], y[:n]


CLASS_ROOT = find_class_root(DATA_DIR)
print("каталог классов:", CLASS_ROOT)

X, y = load_dataset(CLASS_ROOT, max_per_class=MAX_PER_CLASS)

In [ ]:
def describe(name: str, a: np.ndarray) -> None:
    print(f"{name:<8} shape={str(a.shape):<22} dtype={a.dtype!s:<8} {a.nbytes / 2**20:>8.1f} MiB")


describe("X", X)
describe("y", y)

# во что превращался тот же датасет в исходной версии
n = len(X)
print(f"\nдля сравнения, исходные 200x200:")
print(f"  uint8   {n * 200 * 200 / 2**20:>8.1f} MiB")
print(f"  float32 {n * 200 * 200 * 4 / 2**30:>8.2f} GiB (столько занимал бы батч-пайплайн)")
print(f"  пикселей на изображение больше в {200**2 / IMG_SIZE**2:.1f} раза при том же содержании")

## 3. Быстрая проверка данных

Смотрим на баланс классов и на сами картинки: фон чёрный (`0`), штрих светлый.
Это важно для аугментации — заполнять пустоту после сдвига/поворота надо нулями,
а не «отражением» (`fill_mode="reflect"` по умолчанию затащил бы в кадр куски штриха).

In [ ]:
counts = pd.Series(y).value_counts().sort_index()
ratio = counts.max() / counts.min()
print(f"изображений на класс: min={counts.min()}, median={int(counts.median())}, max={counts.max()}")
print(f"дисбаланс max/min = {ratio:.2f}  ->  "
      f"{'веса классов не нужны' if ratio < 1.5 else 'стоит рассмотреть class_weight'}")

fig, axes = plt.subplots(1, 2, figsize=(14, 3.5))
axes[0].bar(counts.index, counts.values, width=1.0)
axes[0].set(title="Число изображений по классам", xlabel="класс", ylabel="кол-во")
axes[1].hist(X[:: max(1, len(X) // 500)].ravel(), bins=50)
axes[1].set(title="Распределение значений пикселей", xlabel="значение", yscale="log")
plt.tight_layout()
plt.show()

In [ ]:
sample_idx = rng.choice(len(X), size=min(24, len(X)), replace=False)

fig, axes = plt.subplots(3, 8, figsize=(12, 5))
for ax, i in zip(axes.ravel(), sample_idx):
    ax.imshow(X[i, :, :, 0], cmap="gray")
    ax.set_title(f"class {y[i]}", fontsize=8)
    ax.axis("off")
plt.tight_layout()
plt.show()

## 4. Диагностика: почему исходная модель стояла на месте

Воспроизводим обе причины на маленькой выборке (быстро, но наглядно). Сравниваем
две сети, отличающиеся **только** нормализацией входа и наличием бутылочного горла
`Dense(10) → Dense(2)`; всё остальное — как в исходнике.

Про ёмкость: формально softmax поверх 2-мерного признака может нарезать плоскость
на 78 выпуклых конусов, так что «математически невозможно» — неверная формулировка.
Практически же обучить такое представление нельзя: градиент проходит через два
подряд сужающихся ReLU (`32 → 10 → 2`), и стоит одному из двух последних нейронов
уйти в отрицательную область — половина представления обнуляется навсегда
(ReLU даёт нулевой градиент при отрицательном входе). Дальше сеть может только
выдавать константу, а лучшая константа для кросс-энтропии — равномерное
распределение, то есть `loss = ln(78)`.

In [ ]:
print(f"ln(78)  = {np.log(NUM_CLASSES):.5f}   <- loss в исходном логе: 4.3565")
print(f"1/78    = {1 / NUM_CLASSES:.5f}   <- accuracy в исходном логе: 0.0126")

In [ ]:
def demo_model(normalize: bool, bottleneck: bool) -> keras.Model:
    """Архитектура исходного ноутбука; два переключателя — две исследуемые причины."""
    inp = keras.Input((IMG_SIZE, IMG_SIZE, 1))
    x = layers.Rescaling(1.0 / 255)(inp) if normalize else inp
    x = layers.Conv2D(32, 3, activation="relu")(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(64, 3, activation="relu")(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Flatten()(x)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.2)(x)
    x = layers.Dense(64, activation="relu")(x)
    x = layers.Dense(32, activation="relu")(x)
    if bottleneck:
        x = layers.Dense(10, activation="relu")(x)
        x = layers.Dense(2, activation="relu", name="bottleneck")(x)
    out = layers.Dense(NUM_CLASSES, activation="softmax")(x)
    return keras.Model(inp, out)


DIAG_N, DIAG_EPOCHS = 4096, 5

if RUN_DIAGNOSTIC:
    idx = rng.choice(len(X), size=min(DIAG_N, len(X)), replace=False)
    Xd, yd = X[idx], y[idx].astype("int32")

    for label, normalize, bottleneck in [
        ("как в исходнике (0..255, есть Dense(2))", False, True),
        ("только нормализация                    ", True, True),
        ("нормализация + без Dense(2)            ", True, False),
    ]:
        keras.utils.set_random_seed(SEED)
        m = demo_model(normalize=normalize, bottleneck=bottleneck)
        m.compile(optimizer=keras.optimizers.Adam(1e-3),
                  loss="sparse_categorical_crossentropy", metrics=["accuracy"])
        m.fit(Xd, yd, epochs=DIAG_EPOCHS, batch_size=32, verbose=0)
        loss, acc = m.evaluate(Xd, yd, verbose=0)

        note = ""
        if bottleneck:
            probe = keras.Model(m.inputs, m.get_layer("bottleneck").output)
            zeros = np.asarray(probe.predict(Xd, verbose=0)) == 0
            note = (f" | нулей на выходе Dense(2): {zeros.mean():>5.1%}"
                    f", полностью мёртвых нейронов: {int(zeros.all(axis=0).sum())}/2")
        print(f"{label}  loss={loss:.4f}  acc={acc:.4f}{note}")

    print(f"\nориентиры: loss ln(78)={np.log(NUM_CLASSES):.4f}, acc 1/78={1 / NUM_CLASSES:.4f}")

Смотреть надо на две вещи. Во-первых, обе конфигурации с `Dense(2)` не выходят
за несколько процентов accuracy, тогда как без горла сеть за те же пять эпох
добирается почти до единицы. Во-вторых, значительная доля выходов бутылочного
горла — нули: это мёртвые ReLU, через которые градиент не проходит. В пределе
оба нейрона умирают полностью, softmax становится равномерным, и получаются ровно
те `loss = ln 78` и `accuracy = 1/78` из исходного лога.

Точные числа зависят от подвыборки и числа эпох; демонстрация специально дешёвая
(подвыборка, 64×64 вместо 200×200, 5 эпох). В исходнике эффект был сильнее:
`Flatten` на 200×200 даёт в 9.8 раза больше входов у `Dense(128)`, предактивации
растут пропорционально, и первый же шаг Adam уводит почти все ReLU в отрицательную
область — отсюда ровное `4.3565` уже на первой эпохе.

Вывод: чинить надо **и** препроцессинг, **и** архитектуру.

## 5. Разбиение train / val / test

Исходная версия делила данные на две части и подавала тест в `validation_data`.
Тогда любое решение «сколько эпох учить / какую модель сохранить» принимается
по тесту, и итоговая цифра завышена. Делаем 70 / 15 / 15 со стратификацией:
val — для early stopping и планировщика LR, test — трогаем один раз в конце.

In [ ]:
def stratify_or_none(labels, test_size):
    """Метки для stratify, либо None: стратификация требует минимум 2 примера на класс."""
    counts = np.bincount(labels)
    counts = counts[counts > 0]
    n_test = int(np.floor(test_size * len(labels)))
    if counts.min() >= 2 and n_test >= len(counts) and len(labels) - n_test >= len(counts):
        return labels
    print(f"стратификация отключена: примеров на класс минимум {counts.min()}, "
          f"классов {len(counts)} — увеличьте MASHTOTS_MAX_PER_CLASS")
    return None


X_train, X_hold, y_train, y_hold = train_test_split(
    X, y, test_size=0.30, random_state=SEED, stratify=stratify_or_none(y, 0.30)
)
X_val, X_test, y_val, y_test = train_test_split(
    X_hold, y_hold, test_size=0.50, random_state=SEED, stratify=stratify_or_none(y_hold, 0.50)
)
del X_hold, y_hold

for name, a, b in [("train", X_train, y_train), ("val", X_val, y_val), ("test", X_test, y_test)]:
    print(f"{name:<6} {len(a):>7} изображений | классов: {len(np.unique(b))}")

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE


def make_ds(images: np.ndarray, labels: np.ndarray, training: bool = False) -> tf.data.Dataset:
    ds = tf.data.Dataset.from_tensor_slices((images, labels.astype("int32")))
    if training:
        ds = ds.shuffle(min(len(images), 10_000), seed=SEED, reshuffle_each_iteration=True)
    return ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)


train_ds = make_ds(X_train, y_train, training=True)
val_ds = make_ds(X_val, y_val)
test_ds = make_ds(X_test, y_test)

## 6. Модель

Ключевые решения:

* **`Rescaling` и аугментация — слоями внутри модели.** `Rescaling` работает и на
  инференсе, поэтому забыть нормализацию при предсказании невозможно (в исходнике
  ровно эта ошибка). `Random*`-слои Keras автоматически выключаются вне обучения.
* **Никакого горизонтального отражения**: буквы зеркально несимметричны, флип
  превратил бы часть классов в мусор. Только небольшие поворот / сдвиг / зум,
  пустота заполняется нулями (чёрный фон датасета).
* **Три блока `Conv-BN-ReLU ×2 → MaxPool → Dropout`.** BatchNorm стабилизирует
  обучение (та же проблема мёртвых ReLU перестаёт быть фатальной), пары свёрток
  дают рецептивное поле 5×5 при меньшем числе параметров, чем одна 5×5.
* **`BatchNormalization(momentum=0.9)` вместо дефолтных `0.99`.** На инференсе BN
  использует не статистики батча, а скользящие средние, которые обновляются как
  `m ← momentum·m + (1-momentum)·батч`. После `k` шагов от инициализации
  (`mean=0, var=1`) остаётся доля `momentum^k`: при `0.99` даже через 400 шагов
  это ещё 1.8 %, а сходится только к ~2000 шагам; при `0.9` хватает сотни
  (`0.9^100 ≈ 3·10⁻⁵`). Пока статистики не сошлись, train-метрики выглядят
  прекрасно, а val-метрики стоят на `1/78` при растущем val-loss. На полном
  датасете (383 шага на эпоху при batch 128) разница почти незаметна, на
  подвыборке — фатальна.
* **Голова без сужений**: `Flatten(8·8·128) → Dense(256) → BN → Dropout(0.4) → Dense(78)`.
  Параметров суммарно ~2.5 М против ~19.9 М в исходнике — и при этом сеть учится.

In [ ]:
BN_MOMENTUM = 0.9  # см. пояснение выше: при 0.99 val-метрики стоят на 1/78 сотни шагов


def build_model(img_size: int = IMG_SIZE, num_classes: int = NUM_CLASSES) -> keras.Model:
    inputs = keras.Input(shape=(img_size, img_size, 1), name="image")

    x = layers.Rescaling(1.0 / 255)(inputs)
    x = layers.RandomRotation(0.03, fill_mode="constant", fill_value=0.0)(x)  # доля от 360°, то есть ±10.8°
    x = layers.RandomTranslation(0.08, 0.08, fill_mode="constant", fill_value=0.0)(x)
    x = layers.RandomZoom(0.10, fill_mode="constant", fill_value=0.0)(x)

    for filters in (32, 64, 128):
        for _ in range(2):
            x = layers.Conv2D(filters, 3, padding="same", use_bias=False)(x)
            x = layers.BatchNormalization(momentum=BN_MOMENTUM)(x)
            x = layers.Activation("relu")(x)
        x = layers.MaxPooling2D(2)(x)
        x = layers.Dropout(0.25)(x)

    x = layers.Flatten()(x)
    x = layers.Dense(256, use_bias=False)(x)
    x = layers.BatchNormalization(momentum=BN_MOMENTUM)(x)
    x = layers.Activation("relu")(x)
    x = layers.Dropout(0.40)(x)
    outputs = layers.Dense(num_classes, activation="softmax", name="probs")(x)

    return keras.Model(inputs, outputs, name="mashtots_cnn")


model = build_model()
model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy", keras.metrics.SparseTopKCategoricalAccuracy(k=3, name="top3")],
)
model.summary()

## 7. Обучение

Вместо фиксированных 5 эпох — колбэки:

* `EarlyStopping(restore_best_weights=True)` останавливает, когда val-accuracy
  перестаёт расти, и возвращает лучшие веса (а не последние, обычно уже
  переобученные);
* `ReduceLROnPlateau` уменьшает LR вдвое на плато — даёт основной прирост в конце;
* `ModelCheckpoint` пишет лучшую модель на диск.

Полезный признак при чтении лога: если **и** train, **и** val стоят на `1/78` —
дело в модели или данных (случай исходного ноутбука). Если train растёт, а val
стоит на `1/78` и val-loss при этом *увеличивается* — почти всегда виноваты слои,
которые ведут себя по-разному в режимах обучения и инференса, то есть BatchNorm
с несошедшимися скользящими статистиками (см. `BN_MOMENTUM` выше).

In [ ]:
callbacks = [
    keras.callbacks.EarlyStopping(
        monitor="val_accuracy", patience=6, restore_best_weights=True, verbose=1
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss", factor=0.5, patience=2, min_lr=1e-5, verbose=1
    ),
    keras.callbacks.ModelCheckpoint(
        MODEL_PATH, monitor="val_accuracy", save_best_only=True, verbose=0
    ),
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks,
    shuffle=False,  # перемешивание уже делает train_ds.shuffle(...)
)

In [ ]:
hist = pd.DataFrame(history.history)
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
hist[["loss", "val_loss"]].plot(ax=axes[0], title="Loss")
axes[0].axhline(np.log(NUM_CLASSES), ls="--", c="r", label="ln(78) — уровень исходной модели")
axes[0].legend()
hist[["accuracy", "val_accuracy"]].plot(ax=axes[1], title="Accuracy")
axes[1].axhline(1 / NUM_CLASSES, ls="--", c="r", label="1/78 — уровень исходной модели")
axes[1].legend()
for ax in axes:
    ax.set_xlabel("эпоха")
plt.tight_layout()
plt.show()

## 8. Честная оценка на отложенном тесте

`test_ds` до этой ячейки не участвовал ни в обучении, ни в выборе модели.

In [ ]:
for name, value in model.evaluate(test_ds, verbose=0, return_dict=True).items():
    print(f"test {name:<10} {value:.4f}")

probs = model.predict(test_ds, verbose=0)
y_pred = probs.argmax(axis=1)

In [ ]:
report = classification_report(y_test, y_pred, output_dict=True, zero_division=0)
per_class = (
    pd.DataFrame(report).T.loc[lambda d: d.index.str.isdigit()]
    .astype({"support": int})
    .sort_values("f1-score")
)
print("10 самых трудных классов (по f1):")
print(per_class.head(10).round(3))

In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=np.arange(NUM_CLASSES))

plt.figure(figsize=(11, 9))
sns.heatmap(cm, cmap="viridis", square=True, cbar_kws={"shrink": 0.7})
plt.title("Confusion matrix (78 классов)")
plt.xlabel("предсказано")
plt.ylabel("истина")
plt.tight_layout()
plt.show()

off = cm.copy()
np.fill_diagonal(off, 0)
pairs = [(a, b) for a, b in np.dstack(
    np.unravel_index(np.argsort(off, axis=None)[::-1], off.shape))[0][:10] if off[a, b] > 0]

if not pairs:
    print("перепутанных пар нет")
else:
    print("Чаще всего путаются (истина -> предсказание, число случаев):")
    for true_c, pred_c in pairs:
        print(f"  {true_c:>2} -> {pred_c:>2} : {off[true_c, pred_c]}")

In [ ]:
wrong = np.flatnonzero(y_pred != y_test)
print(f"ошибок на тесте: {len(wrong)} из {len(y_test)}")

if len(wrong):
    worst = wrong[np.argsort(probs[wrong, y_pred[wrong]])[::-1][:16]]
    fig, axes = plt.subplots(2, 8, figsize=(13, 4))
    for ax, i in zip(axes.ravel(), worst):
        ax.imshow(X_test[i, :, :, 0], cmap="gray")
        ax.set_title(f"{y_test[i]} -> {y_pred[i]}\np={probs[i, y_pred[i]]:.2f}", fontsize=8)
        ax.axis("off")
    for ax in axes.ravel()[len(worst):]:
        ax.axis("off")
    fig.suptitle("Самые уверенные ошибки", y=1.04)
    plt.tight_layout()
    plt.show()

## 9. Предсказание для одного файла

Исходный блок инференса был нерабочим по четырём причинам сразу: пустой путь,
`cv2.cvtColor(img, cv2.IMREAD_GRAYSCALE)` (это не код преобразования цвета),
отсутствие нормализации и отсутствие `argmax`. Здесь одна функция
`preprocess_image` — и она же используется для соревновательного теста,
так что расхождение между обучением и инференсом исключено. Нормализация
живёт внутри модели, поэтому функция возвращает `uint8`.

In [ ]:
def preprocess_image(path, img_size: int = IMG_SIZE) -> np.ndarray:
    img = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise FileNotFoundError(f"не удалось прочитать изображение: {path}")
    if img.shape != (img_size, img_size):
        img = cv2.resize(img, (img_size, img_size), interpolation=cv2.INTER_AREA)
    return img.reshape(1, img_size, img_size, 1)


def predict_letter(path, top_k: int = 3):
    # прямой вызов модели вместо .predict(): для одной картинки быстрее и не
    # провоцирует ретрейсинг tf.function на каждый новый размер батча
    p = np.asarray(model(preprocess_image(path), training=False))[0]
    top = np.argsort(p)[::-1][:top_k]
    return int(top[0]), [(int(c), float(p[c])) for c in top]


demo_path = next(
    p for p in list_images(list_class_dirs(CLASS_ROOT)[0])
    if cv2.imread(str(p), cv2.IMREAD_GRAYSCALE) is not None
)

cls, top = predict_letter(demo_path)
print(f"файл: {demo_path}  (истинный класс по имени папки: {demo_path.parent.name})")
print(f"предсказано: {cls}")
print("top-3:", ", ".join(f"{c} ({p:.3f})" for c, p in top))

## 10. Submission для Kaggle

В архиве соревнования тест лежит либо каталогом картинок, либо таблицей
`new_test.csv`. Поддерживаем оба варианта: если это каталог — читаем файлы тем же
`preprocess_image`; если CSV с ~4096 числовыми колонками — это развёрнутые
пиксели 64×64. Имена колонок берём из `sample_submission.csv`, когда он есть.

In [ ]:
def stack_images(paths) -> np.ndarray:
    """Батч из файлов, без промежуточного списка массивов (тест — это 50 тыс. картинок)."""
    out = np.empty((len(paths), IMG_SIZE, IMG_SIZE, 1), dtype=np.uint8)
    for i, path in enumerate(paths):
        out[i] = preprocess_image(path)[0]
    return out


def load_competition_test(base: Path):
    """(images uint8 (N,64,64,1), ids) для теста соревнования или None, если его нет."""
    for name in ("new_test", "Test", "test"):
        d = base / name
        if d.is_dir():
            files = sorted(p for p in d.rglob("*") if p.suffix.lower() in IMAGE_EXT)
            if files:
                return stack_images(files), [p.stem for p in files]

    for csv_path in (base / "new_test.csv", base / "test.csv"):
        if not csv_path.is_file():
            continue
        df = pd.read_csv(csv_path)
        num = df.select_dtypes(include="number")
        pixel_cols = [c for c in num.columns if num[c].between(0, 255).all()]
        if len(pixel_cols) >= IMG_SIZE * IMG_SIZE:
            # развёрнутые пиксели; берём последние 64*64 колонок, если первая — это id
            pix = num[pixel_cols[-IMG_SIZE * IMG_SIZE:]].to_numpy(dtype=np.uint8)
            imgs = pix.reshape(-1, IMG_SIZE, IMG_SIZE, 1)
            ids = (df[df.columns[0]].tolist() if len(pixel_cols) < len(df.columns)
                   else list(range(len(df))))
            return imgs, ids
        for col in df.columns:
            if df[col].astype(str).str.contains(r"\.(?:png|jpg|jpeg|bmp)$", case=False).any():
                return (stack_images([csv_path.parent / p for p in df[col]]),
                        df[df.columns[0]].tolist())
    return None


test_data = load_competition_test(DATA_DIR)

if test_data is None:
    print("тест соревнования не найден — пропускаем submission")
else:
    imgs, ids = test_data
    preds = model.predict(imgs, batch_size=BATCH_SIZE, verbose=0)

    sample = DATA_DIR / "sample_submission.csv"
    id_col, target_col = ("Id", "Category")
    if sample.is_file():
        id_col, target_col = pd.read_csv(sample, nrows=0).columns[:2]

    submission = pd.DataFrame({id_col: ids, target_col: preds.argmax(axis=1)})
    submission.to_csv("submission.csv", index=False)
    print(f"submission.csv: {len(submission)} строк, колонки {list(submission.columns)}")
    print(f"средняя уверенность: {preds.max(axis=1).mean():.3f}")
    display(submission.head())

## 11. Итоги и что можно улучшить дальше

Исходная сеть выдавала равномерное распределение (`loss = ln 78`, accuracy `1/78`)
из-за двух независимых причин: ненормализованного входа `0..255` и головы
`Dense(10) → Dense(2)` перед 78 классами. После исправления препроцессинга,
архитектуры и протокола валидации сеть обучается, а один прогон стал ещё и
кратно дешевле: 64×64 вместо 200×200 — это ~10× меньше вычислений на изображение
и 2.5 М параметров вместо 19.9 М.

Куда двигаться, если нужна ещё точность:

1. **TTA** — усреднить предсказания по небольшим сдвигам/поворотам (но не по
   отражению: буквы несимметричны).
2. **Ансамбль** 3–5 моделей с разными сидами; на таких задачах даёт +0.5–1 п.п.
3. **Больше эпох + косинусный планировщик LR** вместо `ReduceLROnPlateau`.
4. **Резидуальные блоки** (ResNet-подобный стем) — при 70 тыс. изображений
   более глубокая сеть уже окупается.
5. **`label_smoothing=0.05`** — рукописные буквы содержат объективно неоднозначные
   образцы, смягчение метки уменьшает переуверенность.
6. **`mixed_precision`** на GPU — примерно двукратное ускорение при том же качестве.
7. **Разбор confusion matrix**: пары «заглавная/строчная» одной буквы путаются
   чаще всего; для них может помочь двухступенчатая схема (сначала буква, потом регистр).